### **ACID - PART 4b - Segment object**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2026/04/28

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [1]:
# Import required modules
from importlib.metadata import version
import datetime
import os
from pathlib import Path
import numpy as np
import pandas as pd
import napari
from skimage.transform import resize
# import tifffile
# import matplotlib.pyplot as plt
from acid.utils.listdirNHF import listdirNHF
from acid.utils.get_defaults import default_file_name
from acid.utils.fov_axis_utils import get_fov_ch_shape
from acid.utils.save_image import tifffile_save_ometiff
from acid.image_processing.extract_metadata import extract_ometif_imagej_metadata
from acid.image_processing.make_imagej_metadata import imagej_compatible_metadata_dict



### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modified are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [2]:
# indicate the path to the directory storing the images to segment
# NOTE: this are expected to be the fields of view saved after background
# correction (part4b notebook)
# fov_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\fov_proc"
fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov_proc"

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from either part4a or part3 notebook
# metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"
metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"

# # indicate the path to the directory where outputs will be saved
# NOTE: if the directory does not exist, the pipeline will try to create it
# output_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\seg"
output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\seg"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part4b notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestamp of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"



# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters to select the train set ---
# indicate the name of the column indicating whether the row belongs to train or test set
is_train_column="is_train"

# indicate the value signalling that a row (aka a field of view) belongs to the train set
train_val=1



# # --- parameters used for importing the default metadata dataframe ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False

# separator - used to split the file name and extract the information token with the date
# if metadata_from_file_name is True
metadata_default_separator = '_'

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
# used if metadata_from_file_name is True
metadata_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
# used if metadata_from_file_name is True
metadata_default_date_format='%Y%m%d'

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True


# --- parameters for saving metadata within the saved segmentation mask ---
# image segmentation - separator for metadata entries in the segmentation mask -
# this is the separator used for separating words in the entries of the metadata dictionary
# saved within the segmentation mask.
segmented_img_meta_separator = "_"

# image segmentation - name of processing date in metadata - this is the name
# of the entry in the metadata dictionary to save within the segmentation mask.
# The entry indicates the date when the segmentation was done.
segmented_img_meta_date_name = f"processing{segmented_img_meta_separator}date{segmented_img_meta_separator}yymmdd"

# image segmentation - date format in metadata - this is the format to use for indicating
# the date when the segmentation was done in the metadata saved within the segmentation mask
processing_date_format = '%y%m%d'

# image segmentation - data type in metadata - this is the entry to use for indicating
# the data type of the segmentation mask in the metadata saved within the segmentation mask
segmented_img_meta_dtype_name = 'dtype'

# image segmentation - method name in metadata - this is the entry to use for indicating
# the method used for segmentation in the metadata saved within the segmentation mask
segmented_img_meta_method_name = 'method'

# image segmentation - method in metadata - this is how is called the method
# used for segmentation in the metadata saved within the segmentation mask
segmentation_method_name = "cellpose"

# image segmentation - cellpose version name in metadata - this is the value to use for indicating
# the version of the segmentation method in the metadata saved within the segmentation mask.
segmentation_method_version_name = "version"

# image segmentation - cell pose parameters name in metadata
segmented_img_meta_diameter_name = f"{segmentation_method_name}{segmented_img_meta_separator}diameter"
segmented_img_meta_flow_threshold_name = f"{segmentation_method_name}{segmented_img_meta_separator}flow{segmented_img_meta_separator}threshold"
segmented_img_meta_cellprob_threshold_name = f"{segmentation_method_name}{segmented_img_meta_separator}cellprob{segmented_img_meta_separator}threshold"

# image segmentation processing - name of processing steps in metadata
# this is the entry in the metadata saved within the segmentation mask
# to use for indicating the processing steps applied to the image for segmentation
segmented_img_meta_processing_name = f'processing{segmented_img_meta_separator}steps'

# image segmentation processing - processing steps in metadata
# this is how processing steps applied to the image for segmentation will be indicated in
# the metadata saved within the segmentation mask.
segmented_img_meta_processing_steps = f"median filter nuclei. merge concanavalin and actin by average intensity and median filter the result. downsample channels. stack channels together: pos-0-nuclei, pos-1-merge. cell segmentation. upsample segmentation mask to original size."

# image segmentation preprocessing - downsampling factor name in metadata
# this is the entry to use for indicating the downsampling factor used for segmentation in the metadata saved within the segmentation mask.
segmented_img_meta_downsampling_factor_name = f'downsampling{segmented_img_meta_separator}factor'

# image segmentation preprocessing - nucleus median filter size in metadata
# this is the entry to use for indicating the size of the median filter applied to the nucleus channel before segmentation in the metadata saved within the segmentation mask.
segmented_img_meta_nucleus_med_filter_size_name = f'nucleus{segmented_img_meta_separator}median{segmented_img_meta_separator}filter{segmented_img_meta_separator}size'

# image segmentation preprocessing - concanavalin and actin merge median filter size in metadata
# this is the entry to use for indicating the size of the median filter applied to the image obtained by merging the concanavalin and actin channels before segmentation in the metadata saved within the segmentation mask.
segmented_img_meta_concactin_merge_med_filter_size_name = f'concanavalin{segmented_img_meta_separator}actin{segmented_img_meta_separator}merge{segmented_img_meta_separator}median{segmented_img_meta_separator}filter{segmented_img_meta_separator}size'

# image segmentation postprocessing - upsampling interpolation order name in metadata
segmented_img_meta_resize_order_name = f'upsampling{segmented_img_meta_separator}resize{segmented_img_meta_separator}order'

# image segmentation - name of the raw file entry in the metadata
# this is the string to use to identify the entry in the metadata saved within the preprocessed file
# (aka the illumination corrected file) containing the name of the raw file.
preproc_img_meta_raw_file_name_entry = f'raw{segmented_img_meta_separator}file{segmented_img_meta_separator}name'

# image segmentation - name of the scen entry in the metadata
# this is the string to use to identify the entry in the metadata saved within the preprocessed file
# (aka the illumination corrected file) containing the scene file.
preproc_img_meta_scene_file_name_entry = f'scene{segmented_img_meta_separator}file{segmented_img_meta_separator}name'

# image segmentation - name of the x physical pixel size entry in the metadata
# this is the string to use to identify the entry in the metadata saved within the preprocessed file
# (aka the illumination corrected file) containing the x physical pixel size.
preproc_img_meta_x_physic_px_size_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}x'

# image segmentation - name of the y physical pixel size entry in the metadata
# this is the string to use to identify the entry in the metadata saved within the preprocessed file
# (aka the illumination corrected file) containing the y physical pixel size.
preproc_img_meta_y_physic_px_size_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}y'

# image segmentation - name of the units for the x physical pixel size entry in the metadata
# this is the string to use to identify the entry in the metadata saved within the preprocessed file
# (aka the illumination corrected file) containing the units of the x physical pixel size.
preproc_img_meta_x_physic_px_size_unit_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}unit{segmented_img_meta_separator}x'

# image segmentation - name of the units for the y physical pixel size entry in the metadata
# this is the string to use to identify the entry in the metadata saved within the preprocessed file
# (aka the illumination corrected file) containing the units of the y physical pixel size.
preproc_img_meta_y_physic_px_size_unit_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}unit{segmented_img_meta_separator}y'


# indicate the photometric interpretation to be used when saving the ome.tif files
# NOTE: at the moment, only 'minisblack' has been tested
photometric = 'minisblack'

# indicate whether to save the background function image with ImageJ compatible
save_imagej_compatible = True



# --- parameters for metadata dataframe updating ---
# column name word separator - this is the separator to use for separating the different parts of
# the column names to be added to the metadata dataframe
column_name_separator = "_"


#######################################################################################################
#######################################################################################################
#######################################################################################################
# image segmentation metadata dataframe - computation date column name - this is the name of the column
# to be added to the metadata dataframe to indicate the day when the image segmentation was done.
metadata_df_date_clm_name = f"segmentation{column_name_separator}date"

# image segmentation metadata dataframe - computation date format - this is the format to be used for
# indicating the date when the image segmentation was done. This is used for saving the date in the
# metadata dataframe
metadata_df_meta_date_format = '%y%m%d'

# image segmentation metadata dataframe - segmentation mask column name - this is the name of the column
# to be added to the metadata dataframe to indicate the name used for saving the segmentation file.
metadata_df_file_name_clm_name = f"segmentation{column_name_separator}file{column_name_separator}name"

# image segmentation metadata dataframe - name of the column for the segmentation method in metadata dataframe
metadata_df_method_clm_name = f"segmentation{column_name_separator}method"

# image segmentation metadata dataframe - name of the column for the segmentation method version in
# metadata dataframe
metadata_df_method_version_clm_name = f"segmentation{column_name_separator}method{column_name_separator}version"

# image segmentation metadata dataframe - name of the column for the image segmentation  cell pose
# parameters in metadata
metadata_df_diameter_clm_name = f"{segmentation_method_name}{column_name_separator}diameter"
metadata_df_flow_threshold_clm_name = f"{segmentation_method_name}{column_name_separator}flow{column_name_separator}threshold"
metadata_df_cellprob_threshold_clm_name = f"{segmentation_method_name}{column_name_separator}cellprob{column_name_separator}threshold"

# image segmentation metadata dataframe - name of the column for the downsampling factor in the metadata dataframe
metadata_df_downsampling_factor_clm_name = f"downsampling{column_name_separator}factor"

# image segmentation metadata dataframe - name of the column for the nucleus median filter size in the metadata
# dataframe
metadata_df_nucleus_med_filter_size_name = f"nucleus{column_name_separator}median{column_name_separator}filter{column_name_separator}size"

# image segmentation metadata dataframe - name of the column for the concanavalin and actin merge median
# filter size in metadata dataframe
metadata_df_concactin_merge_med_filter_size_name = f"concactin{column_name_separator}merge{column_name_separator}median{column_name_separator}filter{column_name_separator}size"

# image segmentation metadata dataframe - name of the column for the upsampling interpolation order name in
# metadata dataframe
metadata_df_resize_order_name = f"resize{column_name_separator}order"

# image segmentation metadata dataframe - name of the column for the output data type for the segmentation mask
metadata_df_output_dtype_name = f"output{column_name_separator}dtype"
#######################################################################################################
#######################################################################################################
#######################################################################################################









# --- parameters for image segmentation ---
# image segmentation - corrected file column name - this is the name of the column
# with the name used for saving the illumination corrected file.
illum_correct_df_file_name_clm_name = f"illumination{column_name_separator}correction{column_name_separator}file{column_name_separator}name"

# what is expected to be and what will be used as null value
null_value = np.nan

# channel axis - this is the position of the channel axis in the image to segment.
# It is used to unstack the image into the different channels and select the channels to use for segmentation.
channel_axis = 0

# position of the nucleus channel along the channel axis
# this is used to select the nucleus channel for segmentation
nucleus_position = 0

# position of the concanavalin channel along the channel axis
# this is used to select the concanavalin channel for segmentation
concanavalin_position = 1

# position of the actin channel along the channel axis
# this is used to select the actin channel for segmentation
actin_position = 2

# nucleus median filter size
# this is the size of the median filter to apply to the nucleus channel before segmentation
med_filter_nucleus = 3

# size of the median filter size to apply befor segmentation to the image
# obtained by merging the concanavalin and actin channels
med_filter_concactin_merge = 3

# downsampling factor for segmentation
# this is the factor to use for downsampling (aka binning) the image to segment before segmentation
# NOTE: the factor has to be interpreter as 1/downsampling_factor, meaning that a downsampling_factor of
# 2 means that the image will be resized to half of its original size (aka binned 2x2)
downsampling_factor = 2

# the order of the interpolation used to upsample the segmentation masks after segmentation. Refer to
# skimage.transform.resize documentation https://scikit-image.org/docs/0.25.x/api/skimage.transform.html#skimage.transform.resize
resize_order = 0

# Cell pose parameters
diameter = 75 # the avarage diameter of the cells in pixels. If set to None, cellpose will try to estimate it automatically
flow_threshold=0.4 # the threshold for the flow error. If the flow error is above this value, the cell will not be segmented
cellprob_threshold=0.0 # the threshold for the cell probability. If the cell probability is below this value, the cell will not be segmented


# output data type for the segmentation mask
# either None, if the data type of the segmentation mask should not be changed, or a valid numpy data type
# (e.g. np.uint16) to which the data type of the segmentation mask will be changed before saving.
output_dtype = np.uint16







# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = '_'

# project
project_name = "ACID"

# include indexes when saving pandas dataframes as csv files
save_csv_index = False # if False, the index will not be saved as a separate column in the csv file

# ome suffix - used to save ome.tif files
ome_suffix = ".ome.tif"

# processing metadata dataframe name - savingword
metadata_savingword = "metadata"

# processing metadata dataframe name - file suffix
metadata_file_suffix = f"part{save_file_name_separator}5.csv"

# processing metadata dataframe name - date format
metadata_date_format = '%Y%m%d'

# hyperparameters dataframe name - date format
hyperparameters_date_format = '%Y%m%d-%H%M%S'

# hyperparameters dataframe name - savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters dataframe name - file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}5.csv"



# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = True # if the secondary output directory already exists, do not raise an error



### Create output directory and secondary output directory if they don't exist

##### Output directory stores the segmentation masks
##### Secondary output directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [3]:
# create the output_directory if it doesn't exist
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=exist_ok)

# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)


#### Open the metadata dataframe - this is expected to be the output of part 4b

Run the following cell.

Don't modify the following cell.

In [4]:
# check if using the default metadata data frame (the most recently saved)
if metadata_file_name==None or metadata_file_name.lower()=="default" or metadata_file_name=="":
    
    # import target files in the metadata_directory
    metadata_files = listdirNHF(metadata_directory,
                                target=default_metadata_file_target,
                                exclude=default_metadata_file_exclude)
    
    # get the default metadata file
    metadata_file_name = default_file_name(file_list=metadata_files,
                                           from_file_name=metadata_from_file_name,
                                           directory_path=metadata_directory,
                                           separator=metadata_default_separator,
                                           date_position=metadata_default_date_position,
                                           date_format=metadata_default_date_format,
                                           reverse=metadata_default_reverse)
    
    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# # display the metadata dataframe
# metadata_df


using 20260428_ACID_metadata_part_5.csv as default metadata file


#### Select train data set - NOTE: the metadata_df is updated into a metadata_df which does not contain the test data

Run the following cell.

Don't modify the following cell.

In [5]:
# Select only the train set and update metadata_df
metadata_df = metadata_df[metadata_df[is_train_column] == train_val]

# assert proper selection of train set
assert metadata_df.shape[0] > 0, "No rows in metadata_df belong to the train set."
assert all(metadata_df[is_train_column] == train_val), "Not all rows in metadata_df belong to the train set."

# Display the metadata dataframe
metadata_df

,Unnamed: 0,raw_file_name,scene_name,processing_date_yymmdd,ome_tif_file_name,location,microscope,objective,experiment,condition1,...,segmentation_method,segmentation_method_version,cellpose_diameter,cellpose_flow_threshold,cellpose_cellprob_threshold,downsampling_factor,nucleus_median_filter_size,concactin_merge_median_filter_size,resize_order,output_dtype
0,1,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A2,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,cellpose,4.0.9,75.0,0.4,0.0,2.0,3.0,3.0,0.0,<class 'numpy.uint16'>
1,2,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A3,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,cellpose,4.0.9,75.0,0.4,0.0,2.0,3.0,3.0,0.0,<class 'numpy.uint16'>
2,3,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A4,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A4...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A6,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,H7_DENV2_MOI1_30h_fixed_stained_well1.nd2,A7,251205,H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A7...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,92,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G2,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G2...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
64,93,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G3,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G3...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
65,95,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G5,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G5...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
66,96,H7_DENV2_MOI1_30h_fixed_stained_well5.nd2,G6,251205,H7_DENV2_MOI1_30h_fixed_stained_well5_A07p2_G6...,Center for Integrative Infectious Disease Rese...,Nikon Ti2 - CSU-W1 - spinning disc,Nikon Apochromat Lambda-S 60x/1.40 Oil,A07.2,H7,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Get the shape and the number of channels of the fields of view - NOTE: it is assumed that all fields of view in the dataset have the same shape and number of channels

Run the following cell.

Don't modify the following cell.

In [6]:
# # get the shape of the individual fields of view, the number of channels and the shape of individual channels
# fov_shape, num_channels, shape_of_channels = get_fov_ch_shape(metadata_df,
#                                                               fov_directory,
#                                                               fov_clm=illum_correct_df_file_name_clm_name,
#                                                               channel_axis=channel_axis,
#                                                               null_value=null_value)

import skimage

dummy_d = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov_proc"
dummy_f = "H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A1.ome.tif"
f = skimage.io.imread(os.path.join(dummy_d, dummy_f))
print(f.shape)

fov_shape, num_channels, shape_of_channels = f.shape, f.shape[channel_axis], tuple([s for s in f.shape if s!=f.shape[channel_axis]])
print(fov_shape, num_channels, shape_of_channels)


(5, 1024, 1024)
(5, 1024, 1024) 5 (1024, 1024)


### Import CellPose and CellPose models - NOTE: this is kept separated from previous imports to allow notebook modularity

Run the following cell.

Don't modify the following cell.

In [7]:
# Import required modules
from cellpose import models
from cellpose.io import imread
import torch

# Use gpu if available else cpu
if torch.cuda.is_available():
    use_gpu = True
    print("---------")
    print("GPU is available")
else:
    use_gpu = False
    print("---------")
    print("GPU is not available")

# Import segmentation model
model = models.CellposeModel(gpu=use_gpu)

# get cellpose version
cellpose_version = version(segmentation_method_name)


---------
GPU is not available


### MAIN LOOP
#### 5.1. Preprocess field of views.
#### 5.2. Segment cell and nucleus.
#### 5.2. Save segmentation masks
#### 5.3. Update and save metadata

## **--- --- ---**

Run the following cell.

Don't modify the following cell.

In [ ]:
# from acid.image_processing.segmentation_preprocessing import preprocess_image_for_segmentation
from acid.image_processing.filter_image import median_filter_image
from acid.image_processing.resize_image import downsample_local_mean


# initialize lists to collect processing metadata for updating - this will be used to update the metadata df
segmentation_date_collection = []
segmentation_file_name_collection = []
segmentation_method_collection = []
segmentation_method_version_collection = []
segmentation_diameter_collection = []
segmentation_flow_threshold_collection = []
segmentation_cellprob_threshold_collection = []
segmentation_downsampling_factor_collection = []
segmentation_nucleus_med_filter_size_collection = []
segmentation_concactin_med_filter_size_collection = []
segmentation_resize_order_collection = []
segmentation_output_dtype_collection = []

# iterate through the rows of the metadata dataframe
for file_idx in metadata_df.index:
    print("---------")

    # ---------   ---------
    # OPEN THE FIELD OF VIEW FILE
    # ---------   ---------
    try:
        # get the name of the field of view as ome.tif file
        # field_of_view_file = metadata_df.loc[file_idx, illum_correct_df_file_name_clm_name]
        field_of_view_file = metadata_df.loc[file_idx, 'ome_tif_file_name']

        # form the full path to the field of view file
        img_path = os.path.join(fov_directory,field_of_view_file)

        # open the field of view image as an array
        img = imread(img_path)

        print(f"working on {field_of_view_file}")
    
    except:
        print(f"can't open {field_of_view_file}, skipping image segmentation")

        segmentation_date_collection.append(null_value)
        segmentation_file_name_collection.append(null_value)
        segmentation_method_collection.append(null_value)
        segmentation_method_version_collection.append(null_value)
        segmentation_diameter_collection.append(null_value)
        segmentation_flow_threshold_collection.append(null_value)
        segmentation_cellprob_threshold_collection.append(null_value)
        segmentation_downsampling_factor_collection.append(null_value)
        segmentation_nucleus_med_filter_size_collection.append(null_value)
        segmentation_concactin_med_filter_size_collection.append(null_value)
        segmentation_resize_order_collection.append(null_value)
        segmentation_output_dtype_collection.append(null_value)
        continue

    # ---------   ---------
    # PREPROCESS IMAGE
    # Select nucleus and actin channels
    # filter nucleus using a 10x10 median filtering
    # stack non-filtered nucleus and actin-channel into a new array and filter the channels, individually, using a 3x3 median filtering
    # 2x2 binning
    # NOTE: no dtype conversion is needed as resizing automatically changes image to float
    # ---------   ---------

    # Select nucleus, concanavalin and actin channels
    unstacked_img = np.unstack(img, axis=channel_axis)
    nucleus_channel = unstacked_img[nucleus_position]
    concanavalin_channel = unstacked_img[concanavalin_position]
    actin_channel = unstacked_img[actin_position]

    # merge concanavalin and actin channels
    concactin_merge = np.mean(np.stack([concanavalin_channel, actin_channel], axis=0), axis=0) # axis is hard-coded here as it is irrelevant for the merge operation 

    # median filer nucleus and actin-concanavalin merged channel
    med_nucleus = median_filter_image(nucleus_channel,
                                      size=med_filter_nucleus)
    
    # median filter the merged concanavalin-actin channel
    med_concactin = median_filter_image(concactin_merge,
                                        size=med_filter_concactin_merge)

    # restack nucleus and concanavalin-actin merged channel
    restacked_img = np.stack([med_nucleus, med_concactin], axis=channel_axis)
    
    # downsample the median filtered images
    down_img = downsample_local_mean(restacked_img, factor=downsampling_factor, channel_axis=channel_axis)

    # preproc_img = preprocess_fov(image=actin_channel,
    #                                    channel_axis=channel_axis,
    #                                    med_size=med_size_cell,
    #                                    factor=factor)
    
    preproc_img = down_img.copy() # this is a temporary placeholder

    print("Image preprocessing is done")

    # ---------   ---------
    # SEGMENT CELLS
    # ---------   ---------

    print("Cellular segmentation is beginning. Please wait...")
    
    # cell_masks_i, cell_flows, cell_styles = model.eval(preproc_img, flow_threshold=flow_threshold, cellprob_threshold=cellprob_threshold, diameter=diameter)
    cell_masks_i, cell_flows, cell_styles = np.where(preproc_img[0,...] > 4000, 1,0),"_","_" # this is a temporary placeholder
    
    # re-expand segmentation mask
    cell_masks = resize(cell_masks_i, output_shape=shape_of_channels, order=resize_order)

    print("Cellular segmentation is done")
    
    # ---------
    # UPDATE IMAGE METADATA
    # ---------
    # get field of view metadata
    field_of_view_metadata = extract_ometif_imagej_metadata(img_path)
    
    
    # create a new dictionary to use for including metadata in the saved segmentation mask
    segmentation_metadata_dict = {segmented_img_meta_date_name:datetime.datetime.now().strftime(processing_date_format),
                                  segmented_img_meta_method_name:segmentation_method_name,
                                  segmentation_method_version_name:cellpose_version,
                                  segmented_img_meta_diameter_name:diameter,
                                  segmented_img_meta_flow_threshold_name:flow_threshold,
                                  segmented_img_meta_cellprob_threshold_name:cellprob_threshold,
                                  segmented_img_meta_downsampling_factor_name:downsampling_factor,
                                  segmented_img_meta_nucleus_med_filter_size_name:med_filter_nucleus,
                                  segmented_img_meta_concactin_merge_med_filter_size_name:med_filter_concactin_merge,
                                  segmented_img_meta_resize_order_name:resize_order}


    # ---------
    # SAVE SEGMENTATION MASK
    # ---------

    # change output data type to a format compatible with ImageJ (e.g. uint16) - if needed
    if output_dtype is not None:
        cell_masks = cell_masks.astype(output_dtype)
        print(f"segmentation mask data type has been changed to {output_dtype} for saving")

        # collect the output data type for updating the metadata dataframe
        segmentation_output_dtype_collection.append(output_dtype)

        # update the processing steps for the segmentation metadata dictionary to include the
        # change of data type in the metadata saved within the segmentation mask
        segmented_img_meta_processing_steps += f" change output data type to {output_dtype} for saving."
    else:
        segmentation_output_dtype_collection.append(cell_masks.dtype)

    # update the segmentation metadata dictionary with the processing steps applied to the image for
    # segmentation
    segmentation_metadata_dict[segmented_img_meta_processing_name] = segmented_img_meta_processing_steps

    # update the segmentation metadata dictionary with the data type of the segmentation mask
    segmentation_metadata_dict[segmented_img_meta_dtype_name] = cell_masks.dtype
    
    # convert the dictionary so that entries are distinguishable from default ones
    imagej_segmentation_metadata_dict = imagej_compatible_metadata_dict(segmentation_metadata_dict)

    # carry along important metadata information from the field of view
    for k in field_of_view_metadata.keys():
        for keyword in [preproc_img_meta_raw_file_name_entry, preproc_img_meta_scene_file_name_entry,
                        preproc_img_meta_x_physic_px_size_entry, preproc_img_meta_y_physic_px_size_entry,
                        preproc_img_meta_x_physic_px_size_unit_entry, preproc_img_meta_y_physic_px_size_unit_entry]:
            if keyword in k:
                imagej_segmentation_metadata_dict[k]=field_of_view_metadata[k]
    
    tifffile_save_ometiff(os.path.join(output_directory,field_of_view_file),
                                    data=cell_masks,
                                    imagej=True,
                                    photometric="minisblack",
                                    metadata=imagej_segmentation_metadata_dict)
    
    
    print("Segmentation results have been saved")

    # ---------   ---------
    # UPDATE METADATA DATAFRAME
    # ---------   ---------
    segmentation_date_collection.append(datetime.datetime.now().strftime(metadata_df_meta_date_format))
    segmentation_file_name_collection.append(field_of_view_file)
    segmentation_method_collection.append(segmentation_method_name)
    segmentation_method_version_collection.append(cellpose_version)
    segmentation_diameter_collection.append(diameter)
    segmentation_flow_threshold_collection.append(flow_threshold)
    segmentation_cellprob_threshold_collection.append(cellprob_threshold)
    segmentation_downsampling_factor_collection.append(downsampling_factor)
    segmentation_nucleus_med_filter_size_collection.append(med_filter_nucleus)
    segmentation_concactin_med_filter_size_collection.append(med_filter_concactin_merge)
    segmentation_resize_order_collection.append(resize_order)


print("")
print("Segmentation completed")

# ---------   ---------
# UDDATE METADATA DICTIONARY
# ---------   ---------
metadata_df[metadata_df_date_clm_name] = segmentation_date_collection
metadata_df[metadata_df_file_name_clm_name] = segmentation_file_name_collection
metadata_df[metadata_df_method_clm_name] = segmentation_method_collection
metadata_df[metadata_df_method_version_clm_name] = segmentation_method_version_collection
metadata_df[metadata_df_diameter_clm_name] = segmentation_diameter_collection
metadata_df[metadata_df_flow_threshold_clm_name] = segmentation_flow_threshold_collection
metadata_df[metadata_df_cellprob_threshold_clm_name] = segmentation_cellprob_threshold_collection
metadata_df[metadata_df_downsampling_factor_clm_name] = segmentation_downsampling_factor_collection
metadata_df[metadata_df_nucleus_med_filter_size_name] = segmentation_nucleus_med_filter_size_collection
metadata_df[metadata_df_concactin_merge_med_filter_size_name] = segmentation_concactin_med_filter_size_collection
metadata_df[metadata_df_resize_order_name] = segmentation_resize_order_collection
metadata_df[metadata_df_output_dtype_name] = segmentation_output_dtype_collection

# ---------   ---------
# SAVE THE FINAL METADATA FILE
# ---------   ---------

# save the updated metadata dataframe as a csv file
metadata_df_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df.to_csv(os.path.join(metadata_directory, metadata_df_name), index=save_csv_index)

# print progress update
print("processing metadata saved")
print("finished")




---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A2.ome.tif
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
Cellular segmentation is done
segmentation mask data type has been changed to <class 'numpy.uint16'> for saving
Segmentation results have been saved
---------
working on H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A3.ome.tif
Image preprocessing is done
Cellular segmentation is beginning. Please wait...
Cellular segmentation is done
segmentation mask data type has been changed to <class 'numpy.uint16'> for saving
Segmentation results have been saved
---------
can't open H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A4.ome.tif, skipping image segmentation
---------
can't open H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A6.ome.tif, skipping image segmentation
---------
can't open H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_A7.ome.tif, skipping image segmentation
---------
can't open H7_DENV2_MOI1_30h_fixed_stained_well1_A07p2_B7.ome.tif, 

### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# collect hyperparameters in a dictionary

hyperparameter_dict = {

'fov_directory': fov_directory,
'metadata_directory': metadata_directory,
'output_directory': output_directory,
'metadata_file_name': metadata_file_name,
'is_train_column': is_train_column,
'train_val': train_val,
'default_metadata_file_target': default_metadata_file_target,
'default_metadata_file_exclude': default_metadata_file_exclude,
'metadata_from_file_name': metadata_from_file_name,
'metadata_default_separator': metadata_default_separator,
'metadata_default_date_position': metadata_default_date_position,
'metadata_default_date_format': metadata_default_date_format,
'metadata_default_reverse': metadata_default_reverse,
'segmented_img_meta_date_name': segmented_img_meta_date_name,
'processing_date_format': processing_date_format,
'segmented_img_meta_dtype_name': segmented_img_meta_dtype_name,
'segmentedImage_meta_method_name': segmented_img_meta_method_name,
'segmentation_method_name': segmentation_method_name,
'segmentation_method_version_name': segmentation_method_version_name,
'segmented_img_meta_diameter_name': segmented_img_meta_diameter_name,
'segmented_img_meta_cellprob_threshold_name': segmented_img_meta_cellprob_threshold_name,
'segmented_img_meta_processing_name': segmented_img_meta_processing_name,
'segmented_img_meta_processing_steps': segmented_img_meta_processing_steps,
'segmented_img_meta_downsampling_factor_name': segmented_img_meta_downsampling_factor_name,
'segmented_img_meta_nucleus_med_filter_size_name': segmented_img_meta_nucleus_med_filter_size_name,
'segmented_img_meta_concactin_merge_med_filter_size_name': segmented_img_meta_concactin_merge_med_filter_size_name,
'segmented_img_meta_resize_order_name': segmented_img_meta_resize_order_name,
'preproc_img_meta_raw_file_name_entry': preproc_img_meta_raw_file_name_entry,
'preproc_img_meta_scene_file_name_entry': preproc_img_meta_scene_file_name_entry,
'preproc_img_meta_x_physic_px_size_entry': preproc_img_meta_x_physic_px_size_entry,
'preproc_img_meta_y_physic_px_size_entry': preproc_img_meta_y_physic_px_size_entry,
'preproc_img_meta_x_physic_px_size_unit_entry': preproc_img_meta_x_physic_px_size_unit_entry,
'preproc_img_meta_y_physic_px_size_unit_entry': preproc_img_meta_y_physic_px_size_unit_entry,
'photometric': photometric,
'save_imagej_compatible': save_imagej_compatible,
'column_name_separator': column_name_separator,
'metadata_df_date_clm_name': metadata_df_date_clm_name,
'metadata_df_meta_date_format': metadata_df_meta_date_format,
'metadata_df_file_name_clm_name': metadata_df_file_name_clm_name,
'metadata_df_method_clm_name': metadata_df_method_clm_name,
'metadata_df_method_version_clm_name': metadata_df_method_version_clm_name,
'metadata_df_diameter_clm_name': metadata_df_diameter_clm_name,
'metadata_df_flow_threshold_clm_name': metadata_df_flow_threshold_clm_name,
'metadata_df_cellprob_threshold_clm_name': metadata_df_cellprob_threshold_clm_name,
'metadata_df_downsampling_factor_clm_name': metadata_df_downsampling_factor_clm_name,
'metadata_df_nucleus_med_filter_size_name': metadata_df_nucleus_med_filter_size_name,
'metadata_df_concactin_merge_med_filter_size_name': metadata_df_concactin_merge_med_filter_size_name,
'metadata_df_resize_order_name': metadata_df_resize_order_name,
'metadata_df_output_dtype_name':metadata_df_output_dtype_name,
'illum_correct_df_file_name_clm_name': illum_correct_df_file_name_clm_name,
'null_value': null_value,
'channel_axis': channel_axis,
'nucleus_position': nucleus_position,
'concanavalin_position': concanavalin_position,
'actin_position': actin_position,
'med_filter_nucleus': med_filter_nucleus,
'med_filter_concactin_merge': med_filter_concactin_merge,
'downsampling_factor': downsampling_factor,
'resize_order': resize_order,
'diameter': diameter,
'flow_threshold': flow_threshold,
'cellprob_threshold': cellprob_threshold,
'output_dtype': output_dtype,
'save_file_name_separator': save_file_name_separator,
'project_name': project_name,
'save_csv_index': save_csv_index,
'ome_suffix': ome_suffix,
'metadata_savingword': metadata_savingword,
'metadata_file_suffix': metadata_file_suffix,
'metadata_date_format': metadata_date_format,
'hyperparameters_date_format': hyperparameters_date_format,
'hyperparameters_savingword': hyperparameters_savingword,
'hyperparameters_file_suffix': hyperparameters_file_suffix,
'secondary_output_directory': secondary_output_directory,
'exist_ok': exist_ok

}


# fov_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\fov_proc"
# metadata_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\proc_metadata"
# output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\data\proc\seg"
# metadata_file_name = "default"
# is_train_column="is_train"
# train_val=1
# default_metadata_file_target = ".csv" # files with this string in their name will be selected for the default file selection - if None, all files will be selected
# default_metadata_file_exclude = None # files with this string in their name will be excluded from the default file selection - if None, no files will be excluded
# metadata_from_file_name = False
# metadata_default_separator = '_'
# metadata_default_date_position = 0
# metadata_default_date_format='%Y%m%d'
# metadata_default_reverse = True
# segmented_img_meta_separator = "_"
# segmented_img_meta_date_name = f"processing{segmented_img_meta_separator}date{segmented_img_meta_separator}yymmdd"
# processing_date_format = '%y%m%d'
# segmented_img_meta_dtype_name = 'dtype'
# segmented_img_meta_method_name = 'method'
# segmentation_method_name = "cellpose"
# segmentation_method_version_name = "version"
# segmented_img_meta_diameter_name = f"{segmentation_method_name}{segmented_img_meta_separator}diameter"
# segmented_img_meta_flow_threshold_name = f"{segmentation_method_name}{segmented_img_meta_separator}flow{segmented_img_meta_separator}threshold"
# segmented_img_meta_cellprob_threshold_name = f"{segmentation_method_name}{segmented_img_meta_separator}cellprob{segmented_img_meta_separator}threshold"
# segmented_img_meta_processing_name = f'processing{segmented_img_meta_separator}steps'
# segmented_img_meta_processing_steps = f"median filter nuclei. merge concanavalin and actin by average intensity and median filter the result. downsample channels. stack channels together: pos-0-nuclei, pos-1-merge. cell segmentation. upsample segmentation mask to original size."
# segmented_img_meta_downsampling_factor_name = f'downsampling{segmented_img_meta_separator}factor'
# segmented_img_meta_nucleus_med_filter_size_name = f'nucleus{segmented_img_meta_separator}median{segmented_img_meta_separator}filter{segmented_img_meta_separator}size'
# segmented_img_meta_concactin_merge_med_filter_size_name = f'concanavalin{segmented_img_meta_separator}actin{segmented_img_meta_separator}merge{segmented_img_meta_separator}median{segmented_img_meta_separator}filter{segmented_img_meta_separator}size'
# segmented_img_meta_resize_order_name = f'upsampling{segmented_img_meta_separator}resize{segmented_img_meta_separator}order'
# preproc_img_meta_raw_file_name_entry = f'raw{segmented_img_meta_separator}file{segmented_img_meta_separator}name'
# preproc_img_meta_scene_file_name_entry = f'scene{segmented_img_meta_separator}file{segmented_img_meta_separator}name'
# preproc_img_meta_x_physic_px_size_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}x'
# preproc_img_meta_y_physic_px_size_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}y'
# preproc_img_meta_x_physic_px_size_unit_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}unit{segmented_img_meta_separator}x'
# preproc_img_meta_y_physic_px_size_unit_entry = f'physical{segmented_img_meta_separator}size{segmented_img_meta_separator}unit{segmented_img_meta_separator}y'
# photometric = 'minisblack'
# save_imagej_compatible = True
# column_name_separator = "_"
# metadata_df_date_clm_name = f"segmentation{column_name_separator}date"
# metadata_df_meta_date_format = '%y%m%d'
# metadata_df_file_name_clm_name = f"segmentation{column_name_separator}file{column_name_separator}name"
# metadata_df_method_clm_name = f"segmentation{column_name_separator}method"
# metadata_df_method_version_clm_name = f"segmentation{column_name_separator}method{column_name_separator}version"
# metadata_df_diameter_clm_name = f"{segmentation_method_name}{column_name_separator}diameter"
# metadata_df_flow_threshold_clm_name = f"{segmentation_method_name}{column_name_separator}flow{column_name_separator}threshold"
# metadata_df_cellprob_threshold_clm_name = f"{segmentation_method_name}{column_name_separator}cellprob{column_name_separator}threshold"
# metadata_df_downsampling_factor_clm_name = f"downsampling{column_name_separator}factor"
# metadata_df_nucleus_med_filter_size_name = f"nucleus{column_name_separator}median{column_name_separator}filter{column_name_separator}size"
# metadata_df_concactin_merge_med_filter_size_name = f"concactin{column_name_separator}merge{column_name_separator}median{column_name_separator}filter{column_name_separator}size"
# metadata_df_resize_order_name = f"resize{column_name_separator}order"
# metadata_df_output_dtype_name = f"output{column_name_separator}dtype"
# illum_correct_df_file_name_clm_name = f"illumination{column_name_separator}correction{column_name_separator}file{column_name_separator}name"
# null_value = np.nan
# channel_axis = 0
# nucleus_position = 0
# concanavalin_position = 1
# actin_position = 2
# med_filter_nucleus = 3
# med_filter_concactin_merge = 3
# downsampling_factor = 2
# resize_order = 0
# diameter = 75 # the avarage diameter of the cells in pixels. If set to None, cellpose will try to estimate it automatically
# flow_threshold=0.4 # the threshold for the flow error. If the flow error is above this value, the cell will not be segmented
# cellprob_threshold=0.0 # the threshold for the cell probability. If the cell probability is below this value, the cell will not be segmented
# output_dtype = np.uint16
# save_file_name_separator = '_'
# project_name = "ACID"
# save_csv_index = False # if False, the index will not be saved as a separate column in the csv file
# ome_suffix = ".ome.tif"
# metadata_savingword = "metadata"
# metadata_file_suffix = f"part{save_file_name_separator}5.csv"
# metadata_date_format = '%Y%m%d'
# hyperparameters_date_format = '%Y%m%d-%H%M%S'
# hyperparameters_savingword = "hyperparameters"
# hyperparameters_file_suffix = f"part{save_file_name_separator}5.csv"
# secondary_output_directory = "secondary_output"
# exist_ok = True # if the secondary output directory already exists, do not raise an error
# hyperparameter_series = pd.Series(hyperparameter_dict)
# 76


# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(os.path.join(secondary_output_directory,hyperparameter_saving_name), index=save_csv_index)

print("hyperparameters saved")

hyperparameters saved
